# 09 - Baseline: NLI-Flat (No Retrieval) + External GPT Comparison

Two baselines against our retrieval-augmented pipeline.

---

## Baseline 1 — External: Li et al. (2024) GPT Zero-Shot

Li et al. (2024) *ContraDoc* report GPT zero-shot performance on the **same dataset**:

| Method | Metric | Score |
|--------|--------|-------|
| GPT-4 (Top-k EHR) | Evidence Hit Rate | **70.2%** |
| GPT-3.5 (Top-k EHR) | Evidence Hit Rate | 42.8% |
| GPT-4 (Judge-then-Find R-acc pos) | Recall-accuracy | 35.6% |
| **Our pipeline** (pair-recall, 100-doc subset) | Chunk pair recall | **58.1%** |

Our pipeline reaches 58.1% pair-recall on a 100-doc sample **without fine-tuning**, between GPT-3.5
and GPT-4. The EHR metric (evidence sentence found in top-k) is analogous to our pair-recall
(chunk containing evidence sentence paired with chunk containing ref sentence).

> These numbers are cited directly from Li et al. (2024) and require no additional computation.

---

## Baseline 2 — NLI-Flat (This Notebook)

Run `cross-encoder/nli-deberta-v3-base` on **all intra-document sentence pairs** — no knowledge
graph, no structural filtering, no vector retrieval. This isolates the contribution of our
retrieval stage: if flat NLI performs similarly, retrieval adds no value; if flat NLI is worse
or far noisier, retrieval is load-bearing.

**Input:** `data/processed/ContraDoc/triples.jsonl` (sentence-level, all docs)  
**Gold pairs:** `(gold_evidence_sentence, gold_ref_sentence)` from YES docs in triples.jsonl  
**Output:** threshold table + comparison against pipeline numbers from notebooks 04–06

In [3]:
%pip install sentence-transformers

import json
from collections import defaultdict
from itertools import combinations
from pathlib import Path

import numpy as np
from sentence_transformers import CrossEncoder

TRIPLES_PATH = Path("data/processed/ContraDoc/triples.jsonl")
NLI_MODEL   = "cross-encoder/nli-deberta-v3-base"
BATCH_SIZE  = 32

Note: you may need to restart the kernel to use updated packages.


## Build gold pair set from triples.jsonl

In [4]:
docs = [json.loads(l) for l in TRIPLES_PATH.open(encoding="utf-8")]
yes_docs = [d for d in docs if d["contradiction"] == "YES"]
print(f"Total docs: {len(docs)}  YES: {len(yes_docs)}")

# Gold pairs keyed as (doc_id, evid_sid, ref_sid) — order-normalised
gold_pairs = set()
doc_gold   = defaultdict(set)  # doc_id -> set of (evid_sid, ref_sid)
doc_types  = {}                # doc_id -> list of contra_type strings

for d in yes_docs:
    sents = {s["sentence_id"]: s["source_text"].strip() for s in d["sentences"]}
    evid_id  = d["gold_evidence_sentence_id"]
    ref_ids  = d["gold_ref_sentence_ids"]
    evid_txt = sents.get(evid_id, "")
    contra_types = [t for t in (d.get("contra_type") or "none").split("|") if t]
    doc_types[d["doc_id"]] = contra_types
    for ref_id in ref_ids:
        if evid_txt and sents.get(ref_id, ""):
            key = (d["doc_id"], min(evid_id, ref_id), max(evid_id, ref_id))
            gold_pairs.add(key)
            doc_gold[d["doc_id"]].add((min(evid_id, ref_id), max(evid_id, ref_id)))

print(f"Gold sentence pairs: {len(gold_pairs)}  across {len(doc_gold)} docs")

Total docs: 100  YES: 50
Gold sentence pairs: 37  across 36 docs


## Generate all intra-document sentence pairs

In [5]:
all_pairs = []   # list of dicts: doc_id, sid_a, sid_b, text_a, text_b, is_gold

for d in docs:
    sents = [(s["sentence_id"], s["source_text"].strip())
             for s in d["sentences"] if s["source_text"].strip()]
    for (sid_a, txt_a), (sid_b, txt_b) in combinations(sents, 2):
        lo, hi = min(sid_a, sid_b), max(sid_a, sid_b)
        key    = (d["doc_id"], lo, hi)
        all_pairs.append({
            "doc_id":  d["doc_id"],
            "sid_a":   sid_a,
            "sid_b":   sid_b,
            "text_a":  txt_a,
            "text_b":  txt_b,
            "is_gold": key in gold_pairs,
        })

n_gold_in_flat = sum(1 for p in all_pairs if p["is_gold"])
print(f"Total intra-doc pairs : {len(all_pairs):,}")
print(f"Gold pairs in flat set: {n_gold_in_flat}  (upper bound for flat recall)")

Total intra-doc pairs : 77,147
Gold pairs in flat set: 31  (upper bound for flat recall)


## Load NLI model + score all pairs

In [6]:
model    = CrossEncoder(NLI_MODEL)
id2label = model.config.id2label
label2id = {v.lower(): int(k) for k, v in id2label.items()}
CONTRA_IDX = label2id["contradiction"]
print(f"id2label = {id2label}  contra_idx={CONTRA_IDX}")

text_pairs = [(p["text_a"], p["text_b"]) for p in all_pairs]
scores     = model.predict(text_pairs, batch_size=BATCH_SIZE,
                           show_progress_bar=True, apply_softmax=True)
scores = np.asarray(scores)

contra_scores = scores[:, CONTRA_IDX]
is_gold       = np.array([p["is_gold"] for p in all_pairs], dtype=bool)
doc_ids       = [p["doc_id"] for p in all_pairs]

print(f"Scored {len(scores):,} pairs")

id2label = {0: 'contradiction', 1: 'entailment', 2: 'neutral'}  contra_idx=0


Batches: 100%|██████████| 2411/2411 [04:01<00:00,  9.97it/s]


Scored 77,147 pairs


## Threshold evaluation

In [7]:
n_total_gold = int(is_gold.sum())
gold_doc_ids = {doc_ids[i] for i in range(len(all_pairs)) if is_gold[i]}
n_gold_docs  = len(gold_doc_ids)

print(f"Gold pairs: {n_total_gold}  across {n_gold_docs} docs")
print()
header = f"{'thresh':>6}  {'#pred':>7}  {'TP':>4}  {'FP':>7}  {'FN':>3}  {'Prec':>6}  {'Recall':>6}  {'F1':>6}  {'Doc-R':>11}"
print(header)
print("-" * len(header))
for t in [0.3, 0.5, 0.7, 0.9, 0.95]:
    pred = contra_scores >= t
    tp   = int((pred & is_gold).sum())
    fp   = int((pred & ~is_gold).sum())
    fn   = int((~pred & is_gold).sum())
    prec = tp / max(tp + fp, 1)
    rec  = tp / max(tp + fn, 1)
    f1   = 2 * prec * rec / max(prec + rec, 1e-9)
    doc_tp = {doc_ids[i] for i in np.where(pred & is_gold)[0]}
    doc_r  = len(doc_tp) / max(n_gold_docs, 1)
    print(
        f"{t:>6.2f}  {int(pred.sum()):>7,}  {tp:>4}  {fp:>7,}  {fn:>3}  "
        f"{prec:>5.1%}  {rec:>5.1%}  {f1:>5.1%}  "
        f"{len(doc_tp):>2}/{n_gold_docs:<2} {doc_r:>5.1%}"
    )

Gold pairs: 31  across 31 docs

thresh    #pred    TP       FP   FN    Prec  Recall      F1        Doc-R
------------------------------------------------------------------------
  0.30   16,836    25   16,811    6   0.1%  80.6%   0.3%  25/31 80.6%
  0.50   15,785    25   15,760    6   0.2%  80.6%   0.3%  25/31 80.6%
  0.70   14,826    25   14,801    6   0.2%  80.6%   0.3%  25/31 80.6%
  0.90   13,333    25   13,308    6   0.2%  80.6%   0.4%  25/31 80.6%
  0.95   12,469    25   12,444    6   0.2%  80.6%   0.4%  25/31 80.6%


## Pipeline vs Flat NLI comparison

Numbers from notebooks 04–06 (100-doc sample, chunk-level gold pairs = 23 in retrieved set, 37 total).

| Stage | Candidates | Gold recalled | Recall | Precision |
|-------|-----------|--------------|--------|-----------|
| S-Union (structural only) | 565 | 16/37 | 43.2% | 2.8% |
| Vector@20 ∪ S-Union | 2,451 | 23/37 | 62.2% | 0.9% |
| + NLI-Base ≥ 0.5 | 581 | 22/37 | **59.5%** | 3.8% |
| **Flat NLI ≥ 0.5** (this notebook) | ~77k | _see above_ | — | — |
| GPT-4 zero-shot (Li et al. 2024) | — | — | **70.2%** EHR | — |
| GPT-3.5 zero-shot (Li et al. 2024) | — | — | 42.8% EHR | — |

**Interpreting the comparison:**
- If flat NLI recall ≈ pipeline recall → retrieval stage is not adding coverage; its value is candidate reduction (precision/efficiency)
- If flat NLI recall > pipeline recall → retrieval is dropping gold pairs before NLI sees them (retrieval recall ceiling)
- If flat NLI precision << pipeline precision → retrieval is the key discriminator, not NLI

## Per-type recall (flat NLI @ 0.5)

In [8]:
flagged_at_50 = {
    (all_pairs[i]["doc_id"], 
     min(all_pairs[i]["sid_a"], all_pairs[i]["sid_b"]),
     max(all_pairs[i]["sid_a"], all_pairs[i]["sid_b"]))
    for i in np.where(contra_scores >= 0.5)[0]
}

type_totals  = defaultdict(int)
type_caught  = defaultdict(int)

for key in gold_pairs:
    doc_id = key[0]
    for t in doc_types.get(doc_id, ["unknown"]):
        type_totals[t] += 1
        if key in flagged_at_50:
            type_caught[t] += 1

all_types = sorted(type_totals.keys(), key=lambda x: -type_totals[x])
print(f"Flat NLI @ 0.50:")
print(f"  {'type':30s}  caught  total  recall")
print("  " + "-" * 52)
for t in all_types:
    caught, total = type_caught[t], type_totals[t]
    print(f"  {t:30s}  {caught:>6}  {total:>5}  {caught / max(total, 1):>6.1%}")

Flat NLI @ 0.50:
  type                            caught  total  recall
  ----------------------------------------------------
  Content                             18     23   78.3%
  Numeric                              7     10   70.0%
  Perspective/View/Opinion             4      6   66.7%
  Negation                             4      6   66.7%
  Factual                              2      5   40.0%
  Emotion/Mood/Feeling                 2      4   50.0%
  Causal                               0      2    0.0%
  Relation                             0      1    0.0%


## Summary

Fill in after running:

| Baseline | Recall | Precision @ 0.5 | Candidates |
|----------|--------|-----------------|------------|
| GPT-4 zero-shot (Li et al.) | 70.2% EHR | — | full doc text |
| GPT-3.5 zero-shot (Li et al.) | 42.8% EHR | — | full doc text |
| Pipeline (retrieval + NLI-Base) | 59.5% pair-recall | 3.8% | 2,451 |
| **Flat NLI-Base (this notebook)** | **?%** | **?%** | **~77k** |